# DB got so huge and i want to clean-up it

In [2]:
import os
import duckdb, pandas as pd
from digikala_review_transformer.config import DB_PATH, DATA_INTERIM

# Display settings for long DataFrame values in Jupyter
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

con = duckdb.connect(str(DB_PATH))
cur = con.cursor()

def gb(nbytes):
    return round(nbytes / 1024**3, 3)

# 1) Files on disk: main DB, its WAL, and the backup
files = {
    "main_db": DB_PATH,
    "wal": str(DB_PATH) + ".wal",
    "backup": DATA_INTERIM / "clean_base_backup.duckdb",
}
print(pd.DataFrame([
    {"file": k, "path": str(v), "GB": gb(os.path.getsize(v)) if os.path.exists(v) else None}
    for k, v in files.items()
]))

# 2) Block usage inside the main DB (free blocks = dropped data not yet returned to the OS)
dbs = cur.sql("SELECT * FROM pragma_database_size() WHERE database_name = current_database()").df()
print(dbs.T)
block_size = int(dbs["block_size"].iloc[0])
free_gb = gb(int(dbs["free_blocks"].iloc[0]) * block_size)
print("free (reusable) space inside the file, GB:", free_gb)

# 3) Per-table estimated disk size = distinct storage blocks x block size
tables = cur.sql("""
    SELECT database_name, schema_name, table_name, estimated_size AS est_rows, column_count
    FROM duckdb_tables()
    WHERE NOT internal
""").df()

def table_blocks(db, schema, name):
    q = f'SELECT COUNT(DISTINCT block_id) FROM pragma_storage_info(\'"{db}"."{schema}"."{name}"\') WHERE block_id IS NOT NULL'
    return cur.execute(q).fetchall()[0][0]

tables["est_GB"] = [
    gb(table_blocks(r.database_name, r.schema_name, r.table_name) * block_size)
    for r in tables.itertuples()
]
print(tables.sort_values("est_GB", ascending=False).to_string(index=False))
print("sum of table estimates, GB:", round(tables["est_GB"].sum(), 3))

# 4) Views and attached databases (these can hide extra files or objects)
print(cur.sql("SELECT schema_name, view_name FROM duckdb_views() WHERE NOT internal").df())
print(cur.sql("SELECT database_name, path FROM duckdb_databases() WHERE NOT internal").df())

      file  \
0  main_db   
1      wal   
2   backup   

                                                                                                                    path  \
0                         /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/project.duckdb   
1                     /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/project.duckdb.wal   
2  /home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/data/interim/clean_base_backup.duckdb   

      GB  
0  3.352  
1    NaN  
2  0.297  
                      0
database_name   project
database_size   3.3 GiB
block_size       262144
total_blocks      13731
used_blocks        2935
free_blocks       10796
wal_size        0 bytes
memory_usage    1.7 MiB
memory_limit   24.2 GiB
free (reusable) space inside the file, GB: 2.636
database_name schema_name        table_name  est_rows  column_count  est_GB
      project        main            

In [3]:
from digikala_review_transformer.config import DATA_INTERIM

ARCHIVE = DATA_INTERIM / "dropped_tables_archive"
ARCHIVE.mkdir(exist_ok=True)

TO_ARCHIVE = ["defect_manifest", "body_backup_latin", "dup_plan", "dup_plan_s3",
              "rule_summary", "len_sample"]
TO_DROP = TO_ARCHIVE + ["tmp_dup_groups", "tmp_dup_ids", "tmp_dup_rows", "raw"]

# 1) archive the small tables
for t in TO_ARCHIVE:
    cur.execute(f"COPY {t} TO '{ARCHIVE / (t + '.parquet')}' (FORMAT PARQUET)")

# 2) confirm each archive file has the same row count as its table
for t in TO_ARCHIVE:
    n_db = cur.execute(f"SELECT COUNT(*) FROM {t}").fetchall()[0][0]
    n_pq = cur.execute(f"SELECT COUNT(*) FROM read_parquet('{ARCHIVE / (t + '.parquet')}')").fetchall()[0][0]
    print(t, n_db, n_pq, "OK" if n_db == n_pq else "MISMATCH")

defect_manifest 898769 898769 OK
body_backup_latin 3253 3253 OK
dup_plan 3133 3133 OK
dup_plan_s3 306 306 OK
rule_summary 34 34 OK
len_sample 300000 300000 OK


In [4]:
for t in TO_DROP:
    cur.execute(f"DROP TABLE IF EXISTS {t}")
cur.execute("CHECKPOINT")
print(cur.sql("SELECT table_name, estimated_size FROM duckdb_tables() WHERE NOT internal").df())

   table_name  estimated_size
0  clean_base         6156289


In [1]:
import duckdb
from pathlib import Path
from digikala_review_transformer.config import DB_PATH

src = Path(DB_PATH)
new = src.with_name("project_compact.duckdb")

c = duckdb.connect()
c.execute(f"ATTACH '{src}' AS old (READ_ONLY)")
c.execute(f"ATTACH '{new}' AS new_db")
c.execute("COPY FROM DATABASE old TO new_db")
c.execute("CHECKPOINT new_db")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [1]:
import os, duckdb
from pathlib import Path
from digikala_review_transformer.config import DB_PATH

new_path = Path(DB_PATH)
old_path = new_path.with_name("project_old.duckdb")

c = duckdb.connect()
c.execute(f"ATTACH '{new_path}' AS new_db (READ_ONLY)")
has_old = old_path.exists()
if has_old:
    c.execute(f"ATTACH '{old_path}' AS old_db (READ_ONLY)")

def q(sql):
    return c.execute(sql).fetchall()

results = {}
T = "new_db.main.clean_base"

# 1) Only clean_base should remain
tables = [r[0] for r in q("SELECT table_name FROM duckdb_tables() WHERE database_name='new_db' AND NOT internal")]
results["only clean_base exists"] = (tables == ["clean_base"])
print("tables:", tables)

# 2) Absolute numbers from the earlier reports
rows = q(f"SELECT COUNT(*) FROM {T}")[0][0]
results["total rows = 6,151,947"] = (rows == 6_151_947)
results["NULL step1_rec_label = 893,839"] = (q(f"SELECT COUNT(*) FROM {T} WHERE step1_rec_label IS NULL")[0][0] == 893_839)
results["NULL step1_rate_band_label = 1"] = (q(f"SELECT COUNT(*) FROM {T} WHERE step1_rate_band_label IS NULL")[0][0] == 1)
results["NULL split = 0"] = (q(f"SELECT COUNT(*) FROM {T} WHERE split IS NULL")[0][0] == 0)

split_counts = dict(q(f"SELECT split, COUNT(*) FROM {T} GROUP BY split"))
print("split counts:", split_counts)
results["split sizes match report"] = (split_counts == {"train": 4_306_348, "validation": 922_794, "test": 922_805})

# 3) Corrections still intact (NULL-safe comparison)
n_rec = q(f"SELECT COUNT(*) FROM {T} WHERE step1_rec_label IS DISTINCT FROM recommendation_status")[0][0]
n_band = q(f"SELECT COUNT(*) FROM {T} WHERE step1_rate_band_label IS DISTINCT FROM rating_band")[0][0]
print("rows changed by step 1 -> rec:", n_rec, " band:", n_band)
results["rec corrections = 15,801"] = (n_rec == 15_801)
results["band corrections = 57,242"] = (n_band == 57_242)

# 4) No ID duplicates, no row-counter duplicates
results["id unique"] = (q(f"SELECT COUNT(*) = COUNT(DISTINCT id) FROM {T}")[0][0])
results["_row_idx unique"] = (q(f"SELECT COUNT(*) = COUNT(DISTINCT _row_idx) FROM {T}")[0][0])

# 5) Compare with the old file: same columns and same content per column
if has_old:
    O = "old_db.main.clean_base"
    cols_new = [r[0] for r in q(f"SELECT column_name FROM duckdb_columns() WHERE database_name='new_db' AND table_name='clean_base' ORDER BY column_index")]
    cols_old = [r[0] for r in q(f"SELECT column_name FROM duckdb_columns() WHERE database_name='old_db' AND table_name='clean_base' ORDER BY column_index")]
    results["same columns and order"] = (cols_new == cols_old)

    # per-column fingerprint: sum of hashes (HUGEINT avoids overflow) + null count
    def fingerprint(tbl):
        parts = ", ".join(
            f'SUM(hash("{col}")::HUGEINT) AS h_{i}, COUNT(*) - COUNT("{col}") AS n_{i}'
            for i, col in enumerate(cols_new)
        )
        return q(f"SELECT {parts} FROM {tbl}")[0]
    fp_new, fp_old = fingerprint(T), fingerprint(O)
    bad = [cols_new[i // 2] for i in range(len(fp_new)) if fp_new[i] != fp_old[i]]
    print("columns with different fingerprint:", bad)
    results["all column fingerprints equal old"] = (bad == [])

# 6) File size
size_gb = round(os.path.getsize(new_path) / 1024**3, 3)
print("file size GB:", size_gb)
results["file size < 1 GB"] = (size_gb < 1.0)

# Report
for k, v in results.items():
    print("PASS" if v else "FAIL", "-", k)
print("\nOVERALL:", "PASS" if all(results.values()) else "FAIL")

tables: ['clean_base']
split counts: {'test': 922805, 'validation': 922794, 'train': 4306348}
rows changed by step 1 -> rec: 15801  band: 57242
columns with different fingerprint: []
file size GB: 0.301
PASS - only clean_base exists
PASS - total rows = 6,151,947
PASS - NULL step1_rec_label = 893,839
PASS - NULL step1_rate_band_label = 1
PASS - NULL split = 0
PASS - split sizes match report
PASS - rec corrections = 15,801
PASS - band corrections = 57,242
PASS - id unique
PASS - _row_idx unique
PASS - same columns and order
PASS - all column fingerprints equal old
PASS - file size < 1 GB

OVERALL: PASS


In [3]:
import duckdb
from digikala_review_transformer.config import DB_PATH

c = duckdb.connect(str(DB_PATH), read_only=True)
print(c.sql("SELECT used_blocks, free_blocks, block_size FROM pragma_database_size()").df())
print(c.sql("""
    SELECT column_name, COUNT(DISTINCT block_id) AS blocks,
           ROUND(COUNT(DISTINCT block_id) * 262144 / 1024.0^3, 3) AS GB
    FROM pragma_storage_info('clean_base')
    WHERE block_id IS NOT NULL
    GROUP BY column_name ORDER BY blocks DESC
""").df())

   used_blocks  free_blocks  block_size
0         1325        12406      262144
              column_name  blocks     GB
0                    body     819  0.200
1                   title     129  0.031
2                      id     119  0.029
3              product_id      56  0.014
4             seller_code      53  0.013
5                _row_idx      45  0.011
6                    rate      43  0.010
7                is_buyer      43  0.010
8   recommendation_status      28  0.007
9             rating_band      20  0.005
10        step1_rec_label       8  0.002
11                  split       8  0.002
12  step1_rate_band_label       8  0.002


In [1]:
import duckdb
from pathlib import Path
from digikala_review_transformer.config import DB_PATH

a_path = Path(DB_PATH)                              # original: project.duckdb
b_path = a_path.with_name("project_compact.duckdb") # copy to verify

c = duckdb.connect()
c.execute(f"ATTACH '{a_path}' AS a_db (READ_ONLY)")
c.execute(f"ATTACH '{b_path}' AS b_db (READ_ONLY)")
q = lambda s: c.execute(s).fetchall()

ok = {}

# 1) Same tables
def tables(db):
    return sorted(r[0] for r in q(f"SELECT table_name FROM duckdb_tables() WHERE database_name='{db}' AND NOT internal"))
ta, tb = tables("a_db"), tables("b_db")
print("tables:", ta, tb)
ok["same tables"] = (ta == tb)

# 2) Same columns, types and order
def cols(db):
    return q(f"SELECT column_name, data_type FROM duckdb_columns() WHERE database_name='{db}' AND table_name='clean_base' ORDER BY column_index")
ca, cb = cols("a_db"), cols("b_db")
ok["same columns/types/order"] = (ca == cb)

# 3) Same row counts
na = q("SELECT COUNT(*) FROM a_db.main.clean_base")[0][0]
nb = q("SELECT COUNT(*) FROM b_db.main.clean_base")[0][0]
print("rows:", na, nb)
ok["same row count"] = (na == nb == 6_151_947)

# 4) Exact content check, row by row, matched on _row_idx
names = [name for name, _ in ca]
diff = ", ".join(
    f'SUM(CASE WHEN a."{n}" IS DISTINCT FROM b."{n}" THEN 1 ELSE 0 END) AS "{n}"'
    for n in names
)
res = c.execute(f"""
    SELECT COUNT(*) AS joined_rows, {diff}
    FROM a_db.main.clean_base a JOIN b_db.main.clean_base b ON a._row_idx = b._row_idx
""").df().T
print(res)
joined = int(res.loc["joined_rows"].iloc[0])
ok["every row matched"] = (joined == na)
ok["zero differing cells in every column"] = bool((res.drop(index="joined_rows").iloc[:, 0] == 0).all())

for k, v in ok.items():
    print("PASS" if v else "FAIL", "-", k)
print("OVERALL:", "PASS" if all(ok.values()) else "FAIL")

IOException: IO Error: Cannot open database "/home/alireza/AI-Bootcamp/Project-3-digikala-review/digikala-review-transformer/project_compact.duckdb" in read-only mode: database does not exist

In [1]:
import duckdb
from digikala_review_transformer.config import DATA_INTERIM

c = duckdb.connect(str(DATA_INTERIM / "clean_base_backup_new.duckdb"), read_only=True)
print(c.sql("SELECT COUNT(*) AS rows, COUNT(*) - COUNT(step1_rec_label) AS null_rec, COUNT(*) - COUNT(step1_rate_band_label) AS null_band FROM clean_base").df())
c.close()

      rows  null_rec  null_band
0  6151947    893839          1
